In [3]:
# Cell 1: Setup — clone repo + check environment
import os, subprocess, sys, shutil

# Check GPU (robust)
try:
    nvsmi = shutil.which("nvidia-smi")
    if nvsmi:
        r = subprocess.run([nvsmi, "--query-gpu=name,memory.total", "--format=csv,noheader"],
                           capture_output=True, text=True)
        print(f"GPU: {r.stdout.strip()}")
    else:
        print("nvidia-smi not found — checking torch...")
except Exception:
    pass

# Clone repo if not already present
if not os.path.exists("nst"):
    subprocess.run(["git", "clone", "https://github.com/poolanithinreddy/Neurosymbolic-Transformers.git", "nst"], check=True)
    print("✅ Repo cloned")
elif not os.path.exists("nst/.git"):
    # We might already be inside nst
    pass
else:
    print("✅ Repo already exists")

# cd into nst (handle already being inside)
cwd = os.getcwd()
if os.path.basename(cwd) == "nst" and os.path.exists("main.py"):
    print(f"Already in nst: {cwd}")
elif os.path.exists("nst/main.py"):
    os.chdir("nst")
    print(f"Changed to: {os.getcwd()}")
elif os.path.exists("main.py"):
    print(f"Already in project root: {cwd}")
else:
    print(f"⚠️  Can't find project. CWD: {cwd}, contents: {os.listdir('.')[:10]}")

sys.path.insert(0, os.getcwd())

# Check torch
import torch
print(f"\nPyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    print(f"Device  : MPS (Apple Silicon)")
else:
    print(f"Device  : CPU")

print(f"CWD     : {os.getcwd()}")
print("✅ Setup OK")

nvidia-smi not found — checking torch...
✅ Repo cloned
Changed to: /content/nst

PyTorch : 2.10.0+cpu
CUDA    : False
Device  : CPU
CWD     : /content/nst
✅ Setup OK


In [5]:
# Cell 2: Install dependencies (lightweight — only what's missing)
import subprocess, sys

def pip_install(packages, quiet=True):
    cmd = [sys.executable, "-m", "pip", "install"] + packages
    if quiet:
        cmd.append("-q")
    print(f"Installing: {' '.join(packages)}")
    r = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
    if r.returncode != 0:
        print(f"  ⚠️ Error: {r.stderr.strip()[-200:]}")
    return r.returncode

# Check what we already have
missing = []
for mod, pkg in [("transformers", "transformers"), ("datasets", "datasets"),
                 ("yaml", "pyyaml"), ("sklearn", "scikit-learn"),
                 ("torch", "torch"), ("peft", "peft"), ("accelerate", "accelerate"),
                 ("rank_bm25", "rank-bm25")]:
    try:
        __import__(mod)
    except ImportError:
        missing.append(pkg)

if missing:
    print(f"Missing packages: {missing}")
    pip_install(missing)
else:
    print("All core packages already installed")

# Install the project itself (editable, no deps since we handled them)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps", "-q"],
               capture_output=True, text=True, timeout=60)

# Verify
import transformers, datasets
print(f"\ntransformers : {transformers.__version__}")
print(f"datasets     : {datasets.__version__}")
print("✅ Dependencies OK")

Missing packages: ['rank-bm25']
Installing: rank-bm25

transformers : 5.0.0
datasets     : 4.0.0
✅ Dependencies OK


In [8]:
# Cell 3: Downgrade datasets for FEVER compatibility, then restart kernel
import subprocess, sys

# datasets 4.x removed loading-script support that FEVER needs.
# Install compatible version and restart kernel.
print("Installing datasets==2.21.0 (FEVER compat)...")
r = subprocess.run(
    [sys.executable, "-m", "pip", "install", 
     "datasets==2.21.0", "fsspec>=2023.6,<2025", "huggingface_hub>=0.21,<1.0", "-q"],
    capture_output=True, text=True, timeout=180
)
if r.returncode != 0:
    print(f"Install error: {r.stderr[-500:]}")
else:
    print("✅ Installed. Now RESTART the kernel (Kernel → Restart) and re-run from Cell 4.")
    print("   (Cell 1-3 do NOT need to re-run after restart)")

Installing datasets==2.21.0 (FEVER compat)...
✅ Installed. Now RESTART the kernel (Kernel → Restart) and re-run from Cell 4.
   (Cell 1-3 do NOT need to re-run after restart)


In [7]:
# Cell 4: Verify wiki cache
import os, sys
os.chdir("/content/nst")
sys.path.insert(0, "/content/nst")

# Clear stale module refs so we get updated code
for mod_name in list(sys.modules.keys()):
    if mod_name.startswith("data."):
        del sys.modules[mod_name]

from data.fever_wiki_cache import WikiCache

cache_path = "data/fever_wiki.db"
cache = WikiCache(cache_path)
n = len(cache)
print(f"Wiki cache: {n} pages, {os.path.getsize(cache_path)/1024/1024:.1f} MB")
sample = cache.titles()[:3]
for t in sample:
    sents = cache.lookup(t)
    print(f"  '{t}': {len(sents)} sentences")
cache.close()
print(f"✅ Wiki cache OK ({n} pages)")

Wiki cache: 7000 pages, 11.7 MB
  '"Heroes"_-LRB-David_Bowie_album-RRB-': 9 sentences
  ''Til_Death': 4 sentences
  '...More_Unchartered_Heights_of_Disgrace': 13 sentences
✅ Wiki cache OK (7000 pages)


In [4]:
# Cell 5: FEVER MICRO SMOKE TEST — tiny BERT on CPU (~1 min)
import os, sys, subprocess, logging

os.chdir("/content/nst")
if "/content/nst" not in sys.path:
    sys.path.insert(0, "/content/nst")

# Pull latest (micro smoke config)
r = subprocess.run(["git", "pull", "--ff-only"], capture_output=True, text=True, cwd="/content/nst")
print(f"git pull: {r.stdout.strip()}")

# Clear stale modules
for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.", "logic.", "symbolic."]):
        del sys.modules[mod]

logging.basicConfig(level=logging.INFO, format="%(name)s | %(message)s", force=True)

print("=" * 60)
print("FEVER MICRO SMOKE TEST")
print("  Model: google/bert_uncased_L-2_H-128_A-2 (~4M params)")
print("  50 train / 25 dev / 2 epochs / batch_size 4 / max_len 64")
print("=" * 60 + "\n")

from training.train_fever_nst import train_fever_nst
results = train_fever_nst("configs/fever_micro_smoke.yaml")

print("\n" + "=" * 60)
print("MICRO SMOKE TEST RESULTS:")
print("=" * 60)
if isinstance(results, dict):
    for k, v in results.items():
        if isinstance(v, float):
            print(f"  {k}: {v:.4f}")
        else:
            print(f"  {k}: {v}")
print("\n✅ FEVER micro smoke test completed!")

train_fever | Loading FEVER dataset...
fever_dataset | Loading FEVER from HuggingFace datasets...


git pull: Updating 973068a..f450886
Fast-forward
 configs/fever_micro_smoke.yaml | 40 ++++++++++++++++++++++++++++++++++++++++
 1 file changed, 40 insertions(+)
 create mode 100644 configs/fever_micro_smoke.yaml
FEVER MICRO SMOKE TEST
  Model: google/bert_uncased_L-2_H-128_A-2 (~4M params)
  50 train / 25 dev / 2 epochs / batch_size 4 / max_len 64



fever_dataset |   Using SQLite wiki cache: /content/nst/data/fever_wiki.db (7000 pages)
fever_dataset |   Wiki page map: 7000 pages loaded
fever_dataset |   train: 50 examples (32 with evidence text, 18 without)
fever_dataset |   dev: 25 examples (8 with evidence text, 17 without)
fever_dataset |   train hash: 3fdde0bbcce7a411
fever_dataset |   dev hash: 88547f5236318fb6
train_fever | Using GOLD EVIDENCE mode (Setting A)
fever_nli | Loading model: google/bert_uncased_L-2_H-128_A-2


  FEVER Dataset Statistics

  train: 50 examples
    With gold evidence: 32 (64.0%)
    Label distribution:
      SUPPORTS                 34  (68.0%)
      REFUTES                   6  (12.0%)
      NOT ENOUGH INFO          10  (20.0%)
    Split hash: 3fdde0bbcce7a411

  dev: 25 examples
    With gold evidence: 8 (32.0%)
    Label distribution:
      SUPPORTS                  9  (36.0%)
      REFUTES                   3  (12.0%)
      NOT ENOUGH INFO          13  (52.0%)
    Split hash: 88547f5236318fb6


config.json:   0%|          | 0.00/382 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/17.7M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google/bert_uncased_L-2_H-128_A-2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
fever_nli | Model loaded: google/bert_uncased_L-2_H-128_A-2 (4.4M params)
train_fever | Class weights: [0.4901960790157318, 2.777777671813965, 1.6666666269302368]



  FEVER Training: mode=neural, model=google/bert_uncased_L-2_H-128_A-2
  epochs=2, bs=4, lr=5e-05, device=cpu
  evidence_mode=gold, fp16=False
  total_steps=26, warmup=2

  Epoch 1/2: loss=1.1198 constraint=0.0000


train_fever | Report saved to outputs_fever_micro_smoke/report.json


  Epoch 2/2: loss=1.0864 constraint=0.0000 | dev_acc=0.2000 ECE=0.1535

────────────────────────────────────────
  Post-hoc temperature scaling (dev set)
────────────────────────────────────────
  Learned temperature: T = 1.5003

────────────────────────────────────────
  Final evaluation on dev set
────────────────────────────────────────
  Label Accuracy (GOLD evidence): 0.2000
  ECE: 0.1535
  Brier: 0.6735
    SUPPORTS: acc=0.3333 (n=9)
    REFUTES: acc=0.0000 (n=3)
    NOT ENOUGH INFO: acc=0.1538 (n=13)

  Training complete in 2.6s
  Best dev accuracy: 0.2000
  Output: outputs_fever_micro_smoke

MICRO SMOKE TEST RESULTS:
  mode: neural
  evidence_mode: gold
  model_name: google/bert_uncased_L-2_H-128_A-2
  seed: 42
  epochs: 2
  elapsed_s: 2.6000
  best_dev_acc: 0.2000
  nan_abort: False
  dev: {'accuracy': 0.2, 'ece': 0.15354, 'brier': 0.673474, 'per_label': {'SUPPORTS': {'count': 9, 'accuracy': 0.3333333432674408}, 'REFUTES': {'count': 3, 'accuracy': 0.0}, 'NOT ENOUGH INFO': {'co

In [6]:
# Cell 6: FEVER DeBERTa Smoke Test (100 train, 50 dev, 1 epoch, ~3 min CPU)
import os, sys, logging, time, yaml

os.chdir("/content/nst")
if "/content/nst" not in sys.path:
    sys.path.insert(0, "/content/nst")

# Clear stale modules
for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.", "logic.", "symbolic."]):
        del sys.modules[mod]

logging.basicConfig(level=logging.INFO, format="%(name)s | %(message)s", force=True)

# Create a trimmed config: 100 train (not 200) to fit under 5min CPU timeout
with open("configs/fever_gold_smoke.yaml") as f:
    cfg = yaml.safe_load(f)
cfg["data"]["max_train"] = 100
cfg["data"]["max_dev"] = 50
cfg["io"]["out_dir"] = "outputs_fever_deberta_smoke"
trimmed_path = "/tmp/fever_deberta_smoke.yaml"
with open(trimmed_path, "w") as f:
    yaml.dump(cfg, f)

import torch
device = "cuda" if torch.cuda.is_available() else ("mps" if hasattr(torch.backends, "mps") and torch.backends.mps.is_available() else "cpu")

print("=" * 60)
print("FEVER SMOKE TEST — DeBERTa-v3-base (184M params)")
print(f"  100 train / 50 dev / 1 epoch / device={device}")
if device == "cpu":
    print("  ⚠️  CPU: ~3 min (13 steps × 12s). Expect low accuracy (1 epoch).")
print("=" * 60 + "\n")

t0 = time.time()
from training.train_fever_nst import train_fever_nst
results = train_fever_nst(trimmed_path)
elapsed = time.time() - t0

print("\n" + "=" * 60)
print(f"DeBERTa SMOKE TEST RESULTS ({elapsed:.0f}s):")
print("=" * 60)
if isinstance(results, dict):
    dev = results.get("dev", {})
    print(f"  dev_accuracy: {dev.get('accuracy', 'N/A')}")
    print(f"  dev_ece:      {dev.get('ece', 'N/A')}")
    print(f"  dev_brier:    {dev.get('brier', 'N/A')}")
    for label, stats in dev.get('per_label', {}).items():
        print(f"    {label}: acc={stats.get('accuracy', 0):.3f} (n={stats.get('count', 0)})")
    print(f"  temperature:  {results.get('temperature', 'N/A')}")
    print(f"  elapsed:      {results.get('elapsed_s', elapsed):.1f}s")
print("\n✅ FEVER DeBERTa smoke test completed!")

train_fever | Loading FEVER dataset...
fever_dataset | Loading FEVER from HuggingFace datasets...


FEVER SMOKE TEST — DeBERTa-v3-base (184M params)
  100 train / 50 dev / 1 epoch / device=cpu
  ⚠️  CPU: ~3 min (13 steps × 12s). Expect low accuracy (1 epoch).



fever_dataset |   Using SQLite wiki cache: /content/nst/data/fever_wiki.db (7000 pages)
fever_dataset |   Wiki page map: 7000 pages loaded
fever_dataset |   train: 100 examples (60 with evidence text, 40 without)
fever_dataset |   dev: 50 examples (24 with evidence text, 26 without)
fever_dataset |   train hash: c51d488b6f2faede
fever_dataset |   dev hash: cf72efa91a7ba247
train_fever | Using GOLD EVIDENCE mode (Setting A)
fever_nli | Loading model: microsoft/deberta-v3-base


  FEVER Dataset Statistics

  train: 100 examples
    With gold evidence: 60 (60.0%)
    Label distribution:
      SUPPORTS                 69  (69.0%)
      REFUTES                   9  (9.0%)
      NOT ENOUGH INFO          22  (22.0%)
    Split hash: c51d488b6f2faede

  dev: 50 examples
    With gold evidence: 24 (48.0%)
    Label distribution:
      SUPPORTS                 24  (48.0%)
      REFUTES                  11  (22.0%)
      NOT ENOUGH INFO          15  (30.0%)
    Split hash: cf72efa91a7ba247


Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
fever_nli | Model loaded: microsoft/deberta-v3-base (184.4M params)
train_fever | Class weights: [0.483091801404953, 3.7037036418914795, 1.5151515007019043]



  FEVER Training: mode=neural, model=microsoft/deberta-v3-base
  epochs=1, bs=8, lr=2e-05, device=cpu
  evidence_mode=gold, fp16=False
  total_steps=13, warmup=1

  Epoch 1/1: loss=1.1609 constraint=0.0000 | dev_acc=0.2200 ECE=0.1443

────────────────────────────────────────
  Post-hoc temperature scaling (dev set)
────────────────────────────────────────
  Learned temperature: T = 2.3835

────────────────────────────────────────
  Final evaluation on dev set
────────────────────────────────────────


train_fever | Report saved to outputs_fever_deberta_smoke/report.json


  Label Accuracy (GOLD evidence): 0.2200
  ECE: 0.1443
  Brier: 0.6833
    SUPPORTS: acc=0.0000 (n=24)
    REFUTES: acc=1.0000 (n=11)
    NOT ENOUGH INFO: acc=0.0000 (n=15)

  Training complete in 181.8s
  Best dev accuracy: 0.2200
  Output: outputs_fever_deberta_smoke

DeBERTa SMOKE TEST RESULTS (238s):
  dev_accuracy: 0.22
  dev_ece:      0.144315
  dev_brier:    0.683317
    SUPPORTS: acc=0.000 (n=24)
    REFUTES: acc=1.000 (n=11)
    NOT ENOUGH INFO: acc=0.000 (n=15)
  temperature:  2.3835
  elapsed:      181.8s

✅ FEVER DeBERTa smoke test completed!
